# 03: Test the real backend, then deploy (Harshita)

Notebook 01 tested Whisper on its own. This notebook tests **the actual app code** (`backend/`): upload, audio extraction, background transcription job, and the error cases. Then, optionally, it deploys the app to a Hugging Face Space.

**Before you start**
- `lecture-ai.zip` must be in `MyDrive/lecture_eval/` (same file as for notebook 02).
- Notebook 01 must have been run (it created the clips and excerpts this notebook reuses).
- Run cells in order from the top. Use any Colab runtime (the backend runs on CPU on purpose, like the deployed app).

In [ ]:
!pip -q install fastapi python-multipart google-genai pydantic python-dotenv faster-whisper httpx numpy
import os, sys, glob, time, subprocess
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/lecture_eval"
REPO_ZIP = f"{ROOT}/lecture-ai.zip"
assert os.path.exists(REPO_ZIP), f"Upload lecture-ai.zip to {ROOT} first"
!rm -rf /content/lecture-ai
!unzip -q "{REPO_ZIP}" -d /content      # always unzip fresh so you get the newest code
os.chdir("/content/lecture-ai")          # the backend stores uploads in ./uploads
sys.path.insert(0, "/content/lecture-ai")
os.environ["WHISPER_MODEL"] = "base"     # the model you chose from notebook 01
print("Ready in", os.getcwd())

## Test A: the transcription module on a real 3-minute clip
Uses `backend/transcribe.py` exactly as the app does (CPU, int8).

In [ ]:
from backend import transcribe

clip = sorted(glob.glob(f"{ROOT}/clips/lecture1_*.wav"))[0]
print("Clip:", clip)
t0 = time.time()
lines, segs = transcribe.transcribe_file(clip)
elapsed = time.time() - t0
audio_sec = len(transcribe.load_wav(clip)) / 16000
print(f"Transcribed {audio_sec:.0f}s of audio in {elapsed:.0f}s (includes loading the model once). RTF = {elapsed / audio_sec:.2f}\n")
print("\n".join(lines[:4]))
assert lines and lines[0].startswith("[00:00:"), "Transcript format looks wrong"
print("\nTEST A PASSED")

## Test B: the whole API flow with a 10-minute file
Builds a 10-minute WAV from your excerpt, then does exactly what the Streamlit app does: `/upload`, `/transcribe`, poll `/jobs`.

In [ ]:
from fastapi.testclient import TestClient
from backend import main

src = sorted(glob.glob(f"{ROOT}/excerpts/lecture1_*.wav"))[0]
test_file = "/content/test_10min.wav"
subprocess.run(["ffmpeg", "-y", "-i", src, "-t", "600", test_file], capture_output=True, check=True)

c = TestClient(main.app)
print("health:", c.get("/health").json())

r = c.post("/upload", files={"file": ("test_10min.wav", open(test_file, "rb"))})
print("upload:", r.status_code, r.json())
job_id = r.json()["job_id"]

t0 = time.time()
print("start:", c.post(f"/transcribe/{job_id}").json())
last = -1
while True:
    job = c.get(f"/jobs/{job_id}").json()
    pct = int(job["progress"] * 100)
    if pct != last and pct % 10 == 0:
        print(f"  {pct}%  ({time.time() - t0:.0f}s)")
        last = pct
    if job["status"] != "running":
        break
    time.sleep(2)

total = time.time() - t0
print(f"\nstatus={job['status']}  error={job['error']}  time={total:.0f}s for 600s of audio (RTF {total / 600:.2f})")
print(job["transcript"][:500] if job["transcript"] else "NO TRANSCRIPT")
assert job["status"] == "done"
print("\nTEST B PASSED. Write down the time above for your report/demo planning.")

## Test C: the error messages the user will see
These should all fail *politely*. The last one uses your full ~40-minute lecture, which is over the 15-minute limit.

In [ ]:
print("wrong type:", c.post("/upload", files={"file": ("notes.txt", b"hello")}).json())
print("corrupt   :", c.post("/upload", files={"file": ("fake.mp3", b"this is not audio")}).json())
print("unknown job:", c.get("/jobs/doesnotexist").status_code)

full = glob.glob(f"{ROOT}/audio/lecture1.*")[0]
r = c.post("/upload", files={"file": (os.path.basename(full), open(full, "rb"))})
print("too long  :", r.status_code, r.json())
assert r.status_code == 400
print("\nTEST C PASSED")

## Deploy to a Hugging Face Space (optional, run after A, B and C pass)

**One-time setup**
1. Create a free account at huggingface.co.
2. Go to Settings > Access Tokens > New token, type **Write**. Copy it.
3. In Colab: key icon > Secrets > add `HF_TOKEN` (the token) and make sure `GEMINI_API_KEY` is there too. Turn on Notebook access for both.

The cell creates the Space, uploads the project, and sets the Gemini key as a Space secret. Then open the Space page and watch the **Logs** tab: the first build takes several minutes.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_TOKEN"))
username = api.whoami()["name"]
SPACE = f"{username}/lecture-ai"          # change the name if you like

api.create_repo(SPACE, repo_type="space", space_sdk="docker", exist_ok=True)
api.upload_folder(
    folder_path="/content/lecture-ai", repo_id=SPACE, repo_type="space",
    ignore_patterns=["notebooks/*", "eval/*", "uploads/*", "__pycache__/*", "*.zip", ".env", "*.wav"],
)
api.add_space_secret(SPACE, "GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
print("Uploaded. Open https://huggingface.co/spaces/" + SPACE + " and check the Logs tab.")